# Cold inference demonstration: RF and RF + R34
Use a Conda kernel and set `SOLAR_IMAGES_DIR` and `SOLAR_OUTPUT_DIR`. The notebook verifies and merges the weight shards, then runs all four stages in order. If dependencies are installed in separate environments, set `SOLAR_DENSE_PYTHON`, `SOLAR_SUPPORT_PYTHON`, `SOLAR_SCORE_PYTHON`, and `SOLAR_REFINER_PYTHON` to the corresponding Conda Python executables. Run every cell. Stage logs and completion receipts are saved in the output directory.

In [ ]:
from pathlib import Path
import os, sys, subprocess, csv, time, platform
root = Path.cwd().resolve()
if root.name == "notebooks": root = root.parent
sys.path.insert(0, str(root))
from merge_weights import merge
images = Path(os.environ["SOLAR_IMAGES_DIR"]).resolve()
output = Path(os.environ.get("SOLAR_OUTPUT_DIR", "run_demo")).resolve()
output.mkdir(parents=True, exist_ok=True)
models = output / "models"
started = time.monotonic()
merge(root, models, os.environ.get("SOLAR_RELEASE_RAW_BASE_URL"))
assert images.is_dir()
print("SHARDS_AND_MODELS_SHA_VERIFIED", platform.machine())


In [ ]:
stages = [("dense", "SOLAR_DENSE_PYTHON", "rf", "rf.csv"),
          ("support", "SOLAR_SUPPORT_PYTHON", "rf", "rf.csv"),
          ("score", "SOLAR_SCORE_PYTHON", "rf", "rf.csv"),
          ("refiner", "SOLAR_REFINER_PYTHON", "rf_refiner_r34", "rf_refiner_r34.csv")]
for stage, variable, final, filename in stages:
    interpreter = os.environ.get(variable, sys.executable)
    command = [interpreter, str(root / "scripts/rf_infer.py"), "--images", str(images),
               "--models", str(models), "--cache", str(output / "cache"),
               "--output", str(output / filename), "--stage", stage,
               "--final", final, "--device", "cuda"]
    audit = os.environ.get("SOLAR_AUDIT_ENTRY")
    stage_env = os.environ.copy()
    for key in list(stage_env):
        if key.startswith(("JPY_", "JUPYTER_", "IPYTHON")) or key in (
            "PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONUSERBASE",
            "VIRTUAL_ENV", "CONDA_PREFIX", "CONDA_DEFAULT_ENV", "CONDA_PROMPT_MODIFIER"):
            stage_env.pop(key)
    stage_env["MPLBACKEND"] = "Agg"
    stage_env["PYTHONNOUSERSITE"] = "1"
    stage_env["PATH"] = str(Path(interpreter).resolve().parent) + os.pathsep + stage_env.get("PATH", "")
    if audit:
        command.insert(1, audit)
        stage_env["RF_COLD_AUDIT_OUTPUT"] = str(output / (stage + "_ACCESS.json"))
    with (output / (stage + ".log")).open("w") as log:
        subprocess.run(command, cwd=root, stdout=log, stderr=subprocess.STDOUT, check=True, env=stage_env)
    import json
    (output / (stage + "_NOTEBOOK_DONE.json")).write_text(json.dumps({
        "stage": stage, "elapsed_seconds": time.monotonic() - started,
        "backend": stage_env["MPLBACKEND"], "returncode": 0,
        "removed_kernel_environment": True}))
    print("STAGE_DONE", stage, round(time.monotonic() - started, 2), flush=True)


In [ ]:
for filename in ("rf.csv", "rf_refiner_r34.csv"):
    with (output / filename).open(newline="") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["filament_id", "segmentation_rle"]
        rows = list(reader)
    assert len({row["filament_id"] for row in rows}) == len(rows)
    print(filename, len(rows))
print("TOTAL_SECONDS", time.monotonic() - started)
